<a href="https://colab.research.google.com/github/melissa-04/tubitak-2209a-spatial-stemness-emt/blob/main/01_classification_and_regions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

INPUT_PATH = '/content/drive/MyDrive/Tubitak-2209a/01_processed_data/combined_metadata_spatial.csv'

combined = pd.read_csv(INPUT_PATH)

print('Shape (rows, cols):', combined.shape)
print('Columns:', combined.columns.tolist())
combined.head()

Mounted at /content/drive
Shape (rows, cols): (15611, 13)
Columns: ['barcode', 'nCount_RNA', 'nFeature_RNA', 'patientid', 'subtype', 'Classification', 'region_class', 'exclude_reason', 'array_row', 'array_col', 'pxl_row', 'pxl_col', 'analyze']


,barcode,nCount_RNA,nFeature_RNA,patientid,subtype,Classification,region_class,exclude_reason,array_row,array_col,pxl_row,pxl_col,analyze
0,TACCGATCCAACACTT-1,4073,2071,1142243F,TNBC,Artefact,exclude,Artefact,1.0,1.0,4510.0,12329.0,False
1,GATAAGGGACGATTAG-1,4628,2320,1142243F,TNBC,Artefact,exclude,Artefact,1.0,3.0,4511.0,12601.0,False
2,TGTTGGCTGGCGGAAG-1,5116,2494,1142243F,TNBC,Artefact,exclude,Artefact,1.0,5.0,4512.0,12872.0,False
3,GCGAGGGACTGCTAGA-1,8170,3464,1142243F,TNBC,Artefact,exclude,Artefact,1.0,7.0,4513.0,13144.0,False
4,GCGCGTTTAAATCGTA-1,7534,3345,1142243F,TNBC,Artefact,exclude,Artefact,1.0,9.0,4514.0,13416.0,False


In [2]:

print('1) dtype of analyze:', combined['analyze'].dtype)
print('   unique values   :', combined['analyze'].unique())


print('\n2) NaN per column:')
print(combined.isna().sum())


print('\n3) region_class distribution:')
print(combined['region_class'].value_counts())


print('\n4) Tumor spots per patient:')
tumor_df = combined[combined['region_class'] == 'tumor']
print(tumor_df.groupby('patientid').size())

1) dtype of analyze: bool
   unique values   : [False  True]

2) NaN per column:
barcode               0
nCount_RNA            0
nFeature_RNA          0
patientid             0
subtype               0
Classification       12
region_class          0
exclude_reason    14737
array_row            10
array_col            10
pxl_row              10
pxl_col              10
analyze               0
dtype: int64

3) region_class distribution:
region_class
tumor        11300
non_tumor     3152
exclude        874
in_situ        285
Name: count, dtype: int64

4) Tumor spots per patient:
patientid
1142243F    3627
1160920F    3146
CID4290     2297
CID4465     1131
CID44971     317
CID4535      782
dtype: int64


In [3]:
# Build a per-patient coordinate lookup: (array_row, array_col) -> region_class

grid_df = combined.dropna(subset=['array_row', 'array_col']).copy()
grid_df['array_row'] = grid_df['array_row'].astype(int)
grid_df['array_col'] = grid_df['array_col'].astype(int)

lookup = {}
for patient, group in grid_df.groupby('patientid'):
    lookup[patient] = {
        (row, col): rclass
        for row, col, rclass in zip(group['array_row'], group['array_col'], group['region_class'])
    }

print('Patients in lookup:', list(lookup.keys()))
print('Spots placed on grid (expect 15611 - 10 = 15601):',
      sum(len(d) for d in lookup.values()))

Patients in lookup: ['1142243F', '1160920F', 'CID4290', 'CID4465', 'CID44971', 'CID4535']
Spots placed on grid (expect 15611 - 10 = 15601): 15601


In [4]:
# Validation: duplicate coords, hex parity, coordinate ranges

print('--- 1) Duplicate coordinates within each patient (expect 0) ---')
for patient, group in grid_df.groupby('patientid'):
    n = len(group)
    u = group[['array_row', 'array_col']].drop_duplicates().shape[0]
    print(f'{patient}: {n} spots, {u} unique, duplicates = {n - u}')

print('\n--- 2) Hex parity (array_row + array_col) % 2 — expect a single value ---')
parity = (grid_df['array_row'] + grid_df['array_col']) % 2
print(parity.value_counts())

print('\n--- 3) Coordinate ranges per patient ---')
print(grid_df.groupby('patientid')[['array_row', 'array_col']].agg(['min', 'max']))

--- 1) Duplicate coordinates within each patient (expect 0) ---
1142243F: 4784 spots, 4784 unique, duplicates = 0
1160920F: 4895 spots, 4895 unique, duplicates = 0
CID4290: 2426 spots, 2426 unique, duplicates = 0
CID4465: 1211 spots, 1211 unique, duplicates = 0
CID44971: 1160 spots, 1160 unique, duplicates = 0
CID4535: 1125 spots, 1125 unique, duplicates = 0

--- 2) Hex parity (array_row + array_col) % 2 — expect a single value ---
0    15601
Name: count, dtype: int64

--- 3) Coordinate ranges per patient ---
          array_row     array_col     
                min max       min  max
patientid                             
1142243F          1  77         0  127
1160920F          0  77         0  127
CID4290           4  75        23  127
CID4465           4  76        38  127
CID44971          2  71        14  112
CID4535           3  75         9  101


In [5]:
def hex_distance(dr, dc):
    dr, dc = abs(dr), abs(dc)
    return dr + max(0, (dc - dr) // 2)

def ring_offsets(max_radius):
    offsets = []
    for dr in range(-max_radius, max_radius + 1):
        for dc in range(-2 * max_radius, 2 * max_radius + 1):
            if (dr + dc) % 2 != 0 or (dr == 0 and dc == 0):
                continue
            if hex_distance(dr, dc) <= max_radius:
                offsets.append((dr, dc))
    return offsets

ring2 = ring_offsets(2)
ring3 = ring_offsets(3)

print('2-ring size (expect 18):', len(ring2))
print('3-ring size (expect 36):', len(ring3))

2-ring size (expect 18): 18
3-ring size (expect 36): 36


In [6]:
test_patient = '1142243F'
sample = tumor_df[tumor_df['patientid'] == test_patient].iloc[0]
r0, c0 = int(sample['array_row']), int(sample['array_col'])
patient_grid = lookup[test_patient]

ring1 = [(0, -2), (0, 2), (-1, -1), (-1, 1), (1, -1), (1, 1)]

print(f'Test spot: patient {test_patient}, coord ({r0}, {c0})')
print('\n1-ring (6 immediate neighbours):')
for dr, dc in ring1:
    coord = (r0 + dr, c0 + dc)
    print(f'  {coord}: {patient_grid.get(coord, "EMPTY")}')

def classes_in(grid, r, c, offsets):
    return [grid[(r + dr, c + dc)] for dr, dc in offsets if (r + dr, c + dc) in grid]

c2 = classes_in(patient_grid, r0, c0, ring2)
c3 = classes_in(patient_grid, r0, c0, ring3)
print(f'\n2-ring: {len(c2)}/18 on tissue ->', pd.Series(c2).value_counts().to_dict())
print(f'3-ring: {len(c3)}/36 on tissue ->', pd.Series(c3).value_counts().to_dict())

Test spot: patient 1142243F, coord (3, 3)

1-ring (6 immediate neighbours):
  (3, 1): exclude
  (3, 5): tumor
  (2, 2): exclude
  (2, 4): tumor
  (4, 2): tumor
  (4, 4): tumor

2-ring: 17/18 on tissue -> {'tumor': 10, 'exclude': 7}
3-ring: 26/36 on tissue -> {'tumor': 18, 'exclude': 8}


In [7]:
NON_TUMOR_CLASS = 'non_tumor'
CORE_VALID_MIN = 24

def neighbor_counts(grid, r, c, ring2, ring3):
    classes2 = [grid[(r + dr, c + dc)] for dr, dc in ring2 if (r + dr, c + dc) in grid]
    classes3 = [grid[(r + dr, c + dc)] for dr, dc in ring3 if (r + dr, c + dc) in grid]
    return classes2.count(NON_TUMOR_CLASS), classes3.count(NON_TUMOR_CLASS), len(classes3)

def classify(n_nt2, n_nt3, n_valid3):
    if n_nt2 >= 1:
        return 'front'
    if n_nt3 == 0 and n_valid3 >= CORE_VALID_MIN:
        return 'core'
    return 'uncertain'

tumor_rows = combined[combined['region_class'] == 'tumor']
records = []
for idx, row in tumor_rows.iterrows():
    grid = lookup[row['patientid']]
    r, c = int(row['array_row']), int(row['array_col'])
    n_nt2, n_nt3, n_valid3 = neighbor_counts(grid, r, c, ring2, ring3)
    records.append((idx, n_nt2, n_nt3, n_valid3, classify(n_nt2, n_nt3, n_valid3)))

res = pd.DataFrame(records,
                   columns=['idx', 'n_nontumor_2ring', 'n_nontumor_3ring', 'n_valid_3ring', 'tumor_region']
                   ).set_index('idx')

for col in ['n_nontumor_2ring', 'n_nontumor_3ring', 'n_valid_3ring', 'tumor_region']:
    combined[col] = res[col]

print('Classified tumor spots:', res.shape[0])
print('\ntumor_region counts:')
print(combined['tumor_region'].value_counts(dropna=False))

Classified tumor spots: 11300

tumor_region counts:
tumor_region
core         7783
NaN          4311
uncertain    1785
front        1732
Name: count, dtype: int64


In [8]:
target = {'front': 1732, 'core': 7596, 'uncertain': 1972}
got = combined['tumor_region'].value_counts().to_dict()

print('region      got   target')
for k in ['front', 'core', 'uncertain']:
    print(f'{k:10} {got.get(k, 0):5}   {target[k]}')
print('total tumor:', sum(got.get(k, 0) for k in target))

tum = combined[combined['region_class'] == 'tumor']
print('\nPer-patient:')
print(pd.crosstab(tum['patientid'], tum['tumor_region']))

print('\nSanity - mean non_tumor neighbours per region:')
print(tum.groupby('tumor_region')[['n_nontumor_2ring', 'n_nontumor_3ring']].mean().round(2))

region      got   target
front       1732   1732
core        7783   7596
uncertain   1785   1972
total tumor: 11300

Per-patient:
tumor_region  core  front  uncertain
patientid                           
1142243F      1903   1071        653
1160920F      2571    318        257
CID4290       1917     71        309
CID4465        855     33        243
CID44971       160     58         99
CID4535        377    181        224

Sanity - mean non_tumor neighbours per region:
              n_nontumor_2ring  n_nontumor_3ring
tumor_region                                    
core                      0.00              0.00
front                     3.88              8.40
uncertain                 0.00              1.01


In [9]:
import os

assert combined.shape[0] == 15611, f"Row count changed: {combined.shape[0]}"
assert combined['tumor_region'].notna().sum() == 11300, "Labeled count is not 11300"
assert combined.loc[combined['region_class'] != 'tumor', 'tumor_region'].isna().all(), \
       "Non-tumor rows should be NaN in tumor_region"

print('Pre-save checks passed.')
print('Shape:', combined.shape)
print('Columns:', combined.columns.tolist())

output_dir = '/content/drive/MyDrive/Tubitak-2209a/01_processed_data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'combined_with_regions.csv')
combined.to_csv(output_path, index=False)

print('\nSAVED:', output_path)
reloaded = pd.read_csv(output_path)
print('Reloaded shape:', reloaded.shape)
print('Reloaded tumor_region counts:')
print(reloaded['tumor_region'].value_counts(dropna=False))

Pre-save checks passed.
Shape: (15611, 17)
Columns: ['barcode', 'nCount_RNA', 'nFeature_RNA', 'patientid', 'subtype', 'Classification', 'region_class', 'exclude_reason', 'array_row', 'array_col', 'pxl_row', 'pxl_col', 'analyze', 'n_nontumor_2ring', 'n_nontumor_3ring', 'n_valid_3ring', 'tumor_region']

SAVED: /content/drive/MyDrive/Tubitak-2209a/01_processed_data/combined_with_regions.csv
Reloaded shape: (15611, 17)
Reloaded tumor_region counts:
tumor_region
core         7783
NaN          4311
uncertain    1785
front        1732
Name: count, dtype: int64
